In [1]:
! pip install -U langgraph langchain-anthropic langchain-core langchain-community dotenv langgraph-checkpoint langgraph-checkpoint-sqlite

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, END
import dotenv
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_anthropic import ChatAnthropic
from langchain_community.tools.tavily_search import TavilySearchResults
import operator
import json


C:\Users\ASUS\AppData\Local\Temp\ipykernel_13728\291269279.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [3]:
tool = TavilySearchResults(max_results=4)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_13728\3520374918.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tool = TavilySearchResults(max_results=4)


In [4]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [5]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage],operator.add]

class Agent:

    def __init__(self,model,tools,checkpointer,system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm",self.call_anthropic)
        graph.add_node("action",self.take_action)
        graph.add_conditional_edges("llm",self.action_exists,{True:"action",False:END})
        graph.add_edge("action","llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer,interrupt_before=["action"])
        self.tools = {t.name:t for t in tools }
        self.model = model.bind_tools(tools)

    def call_anthropic(self,state):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {"messages":[message]}

    def action_exists(self,state):
        return len(state["messages"][-1].tool_calls)>0

    def take_action(self,state):
        tool_calls = state["messages"][-1].tool_calls
        results = []
        for t in tool_calls:
            result = self.tools[t["name"]].invoke(t["args"])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        return {"messages":results}



In [6]:
prompt =  """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatAnthropic(model_name="claude-opus-4-8")
abot = Agent(model,[tool],memory,prompt)

In [7]:
messages = [HumanMessage(content="Whats the weather in SF?")]
thread = {"configurable": {"thread_id": "4"}}
for event in abot.graph.stream({"messages":messages},thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content=[{'text': "I'll look up the current weather in San Francisco for you.", 'type': 'text'}, {'id': 'toolu_01WWRYd6dHUzJi61ighCVBqH', 'caller': {'type': 'direct'}, 'input': {'query': 'current weather in San Francisco'}, 'name': 'tavily_search_results_json', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CdPSFAR752bnjbxx7T66X', 'container': None, 'model': 'claude-opus-4-8', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 515, 'output_tokens': 80, 'output_tokens_details': {'thinking_tokens': 0}, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-opus-4-8', 'model_provider': 'anthropic'}, id='lc_run--019f9b0b-4473-7002-92d6-764476e49e94-0', tool_calls=[{'name': 'tavily_search_results_js

In [8]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in SF?', additional_kwargs={}, response_metadata={}), AIMessage(content=[{'text': "I'll look up the current weather in San Francisco for you.", 'type': 'text'}, {'id': 'toolu_01WWRYd6dHUzJi61ighCVBqH', 'caller': {'type': 'direct'}, 'input': {'query': 'current weather in San Francisco'}, 'name': 'tavily_search_results_json', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CdPSFAR752bnjbxx7T66X', 'container': None, 'model': 'claude-opus-4-8', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 515, 'output_tokens': 80, 'output_tokens_details': {'thinking_tokens': 0}, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-opus-4-8', 'model_provider'

In [9]:
abot.graph.get_state(thread).next

('action',)

In [10]:
for event in abot.graph.stream(None,thread):
    for v in event.values():
        print(v)

{'messages': [ToolMessage(content='[{\'title\': \'San Francisco weather in July 2026\', \'url\': \'https://www.weather25.com/north-america/usa/california/san-francisco?page=month&month=July\', \'content\': \'19° / 14°Sunday\\n\\nJul 26\\n\\nSunny\\n\\n0 mm\\n\\n18° / 14°Monday\\n\\nJul 27\\n\\nClear\\n\\n0 mm\\n\\n18° / 14°Tuesday\\n\\nJul 28\\n\\nPatchy rain possible\\n\\n0 mm\\n\\n20° / 13°Wednesday\\n\\nJul 29\\n\\nPatchy rain possible\\n\\n0 mm\\n\\n21° / 14°Thursday\\n\\nJul 30\\n\\nClear\\n\\n0 mm\\n\\n22° / 14°Friday\\n\\nJul 31\\n\\nClear\\n\\n0 mm\\n\\n20° / 14° Next  \\nMonth >>\\n\\n## The average weather in San Francisco in July\\n\\nThe temperatures in San Francisco in July are comfortable with low of 14°C and and high up to 25°C.\\n\\nThere is little to no rain in San Francisco during July, so it’s a lot easier to explore the city. Just remember to dress in warm layers, as it can still get pretty chilly.\\n\\nOur weather forecast can give you a great sense of what weather

In [ ]:
messages = [HumanMessage("Whats the weather in LA?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)
while abot.graph.get_state(thread).next:
    print("\n", abot.graph.get_state(thread),"\n")
    _input = input("proceed?")
    if _input != "y":
        print("aborting")
        break
    for event in abot.graph.stream(None, thread):
        for v in event.values():
            print(v)

{'messages': [AIMessage(content=[{'text': "I'll look up the current weather in Los Angeles for you.", 'type': 'text'}, {'id': 'toolu_01Ud8AGJRGKpU1MYNdVMiCqb', 'caller': {'type': 'direct'}, 'input': {'query': 'current weather in Los Angeles today'}, 'name': 'tavily_search_results_json', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CdPRGcK1DcW64sA9YXK5Q', 'container': None, 'model': 'claude-opus-4-8', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 515, 'output_tokens': 83, 'output_tokens_details': {'thinking_tokens': 0}, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-opus-4-8', 'model_provider': 'anthropic'}, id='lc_run--019f9aff-8edb-7bc3-92c7-0839c9897934-0', tool_calls=[{'name': 'tavily_search_results_

In [73]:
messages = [HumanMessage("Whats the weather in LA?")]
thread = {"configurable": {"thread_id": "new thread"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content=[{'text': "I'll look up the current weather in Los Angeles for you.", 'type': 'text'}, {'id': 'toolu_01XkvqSstfJfzJGrmHrf1NRR', 'caller': {'type': 'direct'}, 'input': {'query': 'current weather in Los Angeles today'}, 'name': 'tavily_search_results_json', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CdPTEkoAJMyXTPMqa1jSP', 'container': None, 'model': 'claude-opus-4-8', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 515, 'output_tokens': 83, 'output_tokens_details': {'thinking_tokens': 0}, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-opus-4-8', 'model_provider': 'anthropic'}, id='lc_run--019f9b17-315e-73a1-ada5-4bc2854626d9-0', tool_calls=[{'name': 'tavily_search_results_

In [74]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}), AIMessage(content=[{'text': "I'll look up the current weather in Los Angeles for you.", 'type': 'text'}, {'id': 'toolu_01XkvqSstfJfzJGrmHrf1NRR', 'caller': {'type': 'direct'}, 'input': {'query': 'current weather in Los Angeles today'}, 'name': 'tavily_search_results_json', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CdPTEkoAJMyXTPMqa1jSP', 'container': None, 'model': 'claude-opus-4-8', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 515, 'output_tokens': 83, 'output_tokens_details': {'thinking_tokens': 0}, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-opus-4-8', 'model_provide

In [75]:
current_values = abot.graph.get_state(thread)

In [76]:
current_values.values['messages'][-1]

AIMessage(content=[{'text': "I'll look up the current weather in Los Angeles for you.", 'type': 'text'}, {'id': 'toolu_01XkvqSstfJfzJGrmHrf1NRR', 'caller': {'type': 'direct'}, 'input': {'query': 'current weather in Los Angeles today'}, 'name': 'tavily_search_results_json', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CdPTEkoAJMyXTPMqa1jSP', 'container': None, 'model': 'claude-opus-4-8', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 515, 'output_tokens': 83, 'output_tokens_details': {'thinking_tokens': 0}, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-opus-4-8', 'model_provider': 'anthropic'}, id='lc_run--019f9b17-315e-73a1-ada5-4bc2854626d9-0', tool_calls=[{'name': 'tavily_search_results_json', 'args':

In [77]:
current_values.values['messages'][-1].tool_calls

[{'name': 'tavily_search_results_json',
  'args': {'query': 'current weather in Los Angeles today'},
  'id': 'toolu_01XkvqSstfJfzJGrmHrf1NRR',
  'type': 'tool_call'}]

In [78]:
_id = current_values.values['messages'][-1].tool_calls[0]['id']
current_values.values['messages'][-1].tool_calls = [
    {'name': 'tavily_search_results_json',
  'args': {'query': 'current weather in Louisiana'},
  'id': _id}
]

In [79]:
abot.graph.update_state(thread, current_values.values)

{'configurable': {'thread_id': 'new thread',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1886c5-1317-6da8-8002-39729225d5bb'}}

In [80]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}), AIMessage(content=[{'text': "I'll look up the current weather in Los Angeles for you.", 'type': 'text'}, {'id': 'toolu_01XkvqSstfJfzJGrmHrf1NRR', 'caller': {'type': 'direct'}, 'input': {'query': 'current weather in Los Angeles today'}, 'name': 'tavily_search_results_json', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CdPTEkoAJMyXTPMqa1jSP', 'container': None, 'model': 'claude-opus-4-8', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 515, 'output_tokens': 83, 'output_tokens_details': {'thinking_tokens': 0}, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-opus-4-8', 'model_provide

In [ ]:
for event in abot.graph.stream(None, thread):
    for v in event.values():
        print(v)